# Chapter 01: Training Foundations

**Math companion:** [Beginner mathematical view](00_math_intuition.ipynb). Study its relevant sections alongside these lessons.

[Course index](../README.md) · [Next](../02_batches_and_epochs/README.md)

**Goal:** Understand parameters, predictions, loss, gradients, learning rate, and repeated updates.

**Study order:** start here and run lessons in order. The installation cell is optional
when PyTorch is already available. Start with `import torch` in lesson 02.

Before each experiment, record your prediction; afterwards, explain the output.
Related revision checkpoints: **1–4** in the
[revision notebook](../revision/01_training_basics_revision.ipynb).

## Contents

- [01. Install PyTorch (optional)](#lesson-01)
- [02. Import PyTorch](#lesson-02)
- [03. Create four training examples](#lesson-03)
- [04. Make predictions with weight and bias](#lesson-04)
- [05. Measure prediction error with mean squared loss](#lesson-05)
- [06. Calculate gradients without changing parameters](#lesson-06)
- [07. Apply one parameter update](#lesson-07)
- [08. Compare learning rates from the same starting point](#lesson-08)
- [09. Recalculate gradients after an update](#lesson-09)
- [10. Repeat the full-batch training loop for 10 steps](#lesson-10)
- [11. Check what happens over 100 steps](#lesson-11)


## Lesson 01: Install PyTorch (optional)

<a id="lesson-01"></a>



Use this cell only when setting up the notebook environment. `%pip` installs into
the notebook kernel's Python environment. Restart the kernel if requested after installation.
This cell does not train a model; skip it if the next cell imports PyTorch successfully.


In [5]:
# Optional setup: these commands install packages, not model parameters.
%pip install --upgrade pip
%pip install torch


Defaulting to user installation because normal site-packages is not writeable
  Using cached pip-26.2.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.0.1
    Uninstalling pip-25.0.1:
      Successfully uninstalled pip-25.0.1
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Lesson 02: Import PyTorch

<a id="lesson-02"></a>



Import PyTorch and print the installed version. A tensor is PyTorch's container
for numbers. We will use tensors for both data and adjustable parameters.


In [1]:
import torch

# Confirm that the notebook kernel can use PyTorch.
print("PyTorch version:", torch.__version__)


PyTorch version: 2.6.0+cu124


## Lesson 03: Create four training examples

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#prediction).

<a id="lesson-03"></a>



`x` holds the inputs and `y` holds the matching correct answers in the same order.
The decimal points create floating-point tensors, which we use for gradient-based training.
`torch.Size([4])` means the tensor is a one-dimensional sequence containing four values.


In [2]:
# Each position in x matches the same position in y.
x = torch.tensor([0., 1., 2., 3.])
y = torch.tensor([2., 5., 8., 11.])

print("Inputs:", x)
print("Correct answers:", y)
print("Shape:", x.shape)


Inputs: tensor([0., 1., 2., 3.])
Correct answers: tensor([ 2.,  5.,  8., 11.])
Shape: torch.Size([4])


## Lesson 04: Make predictions with weight and bias

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#prediction).

<a id="lesson-04"></a>



The model has **two parameters**: one weight and one bias. Both are shared across all
four examples. `requires_grad=True` asks PyTorch to track calculations involving these
parameters so it can calculate gradients later; it does not train them yet.

**Predict first:** with weight `1` and bias `0`, the predictions should be `[0, 1, 2, 3]`.
`detach()` is used only for displaying values without their gradient-tracking connection.


In [3]:
# Reset the model to our starting parameters.
weight = torch.tensor(1., requires_grad=True)
bias = torch.tensor(0., requires_grad=True)

# The same weight and bias are applied to every input.
predictions = weight * x + bias
print("Predictions:", predictions.detach().tolist())


Predictions: [0.0, 1.0, 2.0, 3.0]


## Lesson 05: Measure prediction error with mean squared loss

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#loss).

<a id="lesson-05"></a>



Subtract each correct answer from its prediction, square the error, then average
the four squared errors. Squaring makes both positive and negative errors contribute
positively to loss. A zero loss here means every prediction matches its target.

At the starting parameters, squared errors are `[4, 16, 36, 64]` and their mean is `30`.
Calculating loss alone does not update parameters.


In [4]:
# Compute one squared error for each example.
squared_errors = (predictions - y) ** 2

# Combine the four errors into one mean loss.
loss = squared_errors.mean()
print("Squared errors:", squared_errors.detach().tolist())
print("Mean loss:", loss.item())


Squared errors: [4.0, 16.0, 36.0, 64.0]
Mean loss: 30.0


## Lesson 06: Calculate gradients without changing parameters

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#gradients).

<a id="lesson-06"></a>



`loss.backward()` calculates gradients and stores them in `.grad`. It does **not**
change the weight or bias. PyTorch accumulates gradients, so clear old gradients before
calculating fresh ones. Assigning `.grad = None` preserves the learned parameter values.

At weight `1`, bias `0`, the gradients are **weight −20, bias −10**. A negative gradient
means increasing that parameter would reduce loss locally. A sufficiently large update
can still overshoot. Rebuilding predictions and loss creates a fresh calculation graph.


In [5]:
# Clear stored gradients; the learned weight and bias stay unchanged.
weight.grad = None
bias.grad = None

# Recalculate the loss from the current parameters using a fresh graph.
predictions = weight * x + bias
loss = ((predictions - y) ** 2).mean()

# Calculate gradients, without performing an update.
loss.backward()
print("Weight gradient:", weight.grad.item())
print("Bias gradient:", bias.grad.item())


Weight gradient: -20.0
Bias gradient: -10.0


## Lesson 07: Apply one parameter update

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#update).

<a id="lesson-07"></a>



The update rule is **new parameter = old parameter − learning rate × gradient**.
The learning rate scales the update; it does not change the gradient calculated at a
fixed set of parameters. Subtracting a negative gradient increases the parameter.

This experiment resets to weight `1`, bias `0` and recalculates gradients so rerunning
this cell always demonstrates exactly one update. With learning rate `0.05`, expect
weight `2`, bias `0.5`, and loss to fall from `30` to `10.25`.

`torch.no_grad()` stops PyTorch from tracking the update itself. The parameters still change.


In [6]:
# Reset so repeated runs of this cell start from the same position.
weight = torch.tensor(1., requires_grad=True)
bias = torch.tensor(0., requires_grad=True)
learning_rate = 0.05

# Calculate fresh gradients at this starting point.
predictions = weight * x + bias
loss = ((predictions - y) ** 2).mean()
loss.backward()
before = loss.item()

with torch.no_grad():
    # Apply the gradients to both parameters without tracking the update.
    weight -= learning_rate * weight.grad
    bias -= learning_rate * bias.grad

    # Measure predictions and loss again AFTER the update.
    new_predictions = weight * x + bias
    after = ((new_predictions - y) ** 2).mean().item()

print("New weight:", weight.item())
print("New bias:", bias.item())
print("New predictions:", new_predictions.tolist())
print("Loss before:", before)
print("Loss after:", after)


New weight: 2.0
New bias: 0.5
New predictions: [0.5, 2.5, 4.5, 6.5]
Loss before: 30.0
Loss after: 10.25


## Lesson 08: Compare learning rates from the same starting point

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#update).

<a id="lesson-08"></a>



Change only the learning rate: every run starts at weight `1`, bias `0`, uses all four
examples, and makes one update. The starting gradients are identical across these runs.

**Predict first:** which rate will improve loss most after one update? Rate `0.01`
improves slowly, `0.05` improves more, and `0.50` overshoots, raising loss to `305`.
The conclusion is that step size matters; the smallest rate is not always the best choice.
The function uses local `w` and `b`, so it does not change the earlier `weight` and `bias`.


In [7]:
def one_step(lr):
    """Reset a model, apply one full-batch update, and return its results."""
    # A fresh start makes learning-rate comparisons reproducible.
    w = torch.tensor(1., requires_grad=True)
    b = torch.tensor(0., requires_grad=True)

    initial_loss = ((w * x + b - y) ** 2).mean()
    initial_loss.backward()

    with torch.no_grad():
        # lr scales the gradients; it does not calculate them.
        w -= lr * w.grad
        b -= lr * b.grad
        final_loss = ((w * x + b - y) ** 2).mean()

    return w.item(), b.item(), initial_loss.item(), final_loss.item()

print("rate   weight   bias   loss before   loss after")
for rate in [0.01, 0.05, 0.5]:
    w, b, old, new = one_step(rate)
    print(f"{rate:<6.2f} {w:<8.2f} {b:<6.2f} {old:<13.2f} {new:.2f}")


rate   weight   bias   loss before   loss after
0.01   1.20     0.10   30.00         25.21
0.05   2.00     0.50   30.00         10.25
0.50   11.00    5.00   30.00         305.00


## Lesson 09: Recalculate gradients after an update

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#gradients).

<a id="lesson-09"></a>



An update changes predictions and errors, so the next step needs new gradients.
This cell starts at the first update's result, weight `2`, bias `0.5`, rather than
reusing the initial gradients −20 and −10. Expect **weight gradient −11.5, bias gradient −6**.
With learning rate `0.05`, the next weight would be `2.575`, not `2.525`.
This cell calculates the new gradients without applying another update.


In [8]:
# Recreate the first update's result to make this demonstration repeatable.
weight = torch.tensor(2., requires_grad=True)
bias = torch.tensor(0.5, requires_grad=True)

# Clear gradients before calculating them at the new parameter values.
weight.grad = None
bias.grad = None
predictions = weight * x + bias
loss = ((predictions - y) ** 2).mean()
loss.backward()

print("Weight gradient:", weight.grad.item())
print("Bias gradient:", bias.grad.item())


Weight gradient: -11.5
Bias gradient: -6.0


## Lesson 10: Repeat the full-batch training loop for 10 steps

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#update).

<a id="lesson-10"></a>



Reset the parameters, then repeat predict → loss → gradients → update ten times.
Each printed row reports loss **after** the update. All four examples contribute to
each update, so batch size is `4`. One step is also one epoch in this particular loop.

The learning rate stays fixed, but each update uses new gradients. Expect loss to
decrease by different amounts, reaching about `0.1513`. Weight can exceed `3` while
bias remains below `2`: the two parameters jointly reduce prediction errors.


In [9]:
# Reset once, before training; preserve learned values between loop iterations.
weight = torch.tensor(1., requires_grad=True)
bias = torch.tensor(0., requires_grad=True)
learning_rate = 0.05

for step in range(10):
    # Clear previous gradients, not the learned parameters.
    weight.grad = None
    bias.grad = None

    # Use all four examples to calculate one mean loss.
    predictions = weight * x + bias
    loss = ((predictions - y) ** 2).mean()
    loss.backward()

    with torch.no_grad():
        # One update per iteration, after combining all four errors.
        weight -= learning_rate * weight.grad
        bias -= learning_rate * bias.grad
        updated_loss = ((weight * x + bias - y) ** 2).mean()

    print(
        f"Step {step + 1:2d} | "
        f"weight={weight.item():.4f} | "
        f"bias={bias.item():.4f} | "
        f"loss={updated_loss.item():.4f}"
    )


Step  1 | weight=2.0000 | bias=0.5000 | loss=10.2500
Step  2 | weight=2.5750 | bias=0.8000 | loss=3.6022
Step  3 | weight=2.9037 | bias=0.9837 | loss=1.3586
Step  4 | weight=3.0899 | bias=1.0998 | loss=0.5959
Step  5 | weight=3.1934 | bias=1.1763 | loss=0.3314
Step  6 | weight=3.2493 | bias=1.2297 | loss=0.2348
Step  7 | weight=3.2776 | bias=1.2693 | loss=0.1951
Step  8 | weight=3.2900 | bias=1.3008 | loss=0.1749
Step  9 | weight=3.2934 | bias=1.3272 | loss=0.1618
Step 10 | weight=3.2916 | bias=1.3505 | loss=0.1513


## Lesson 11: Check what happens over 100 steps

**Mathematical view:** [Related explanation and worked example](00_math_intuition.ipynb#update).

<a id="lesson-11"></a>



**Our hypothesis:** loss might reach its minimum and then increase if training keeps
moving past the solution. Test that hypothesis by resetting and running 100 steps.
Here, at learning rate `0.05`, loss decreases at every step; weight approaches `3`
and bias approaches `2`.

A fixed learning rate does not imply a fixed update. Smaller gradients produce smaller
changes. If this model's squared-error loss is exactly zero, its gradients are zero,
so these plain gradient-descent updates stop. Too large a learning rate can still overshoot.
These are training-loss results on the same four examples; they do not measure generalization.


In [10]:
# Reset for a separate experiment with a longer training duration.
w = torch.tensor(1., requires_grad=True)
b = torch.tensor(0., requires_grad=True)
learning_rate = 0.05
with torch.no_grad():
    previous_loss = ((w * x + b - y) ** 2).mean().item()
increases = []

for step in range(1, 101):
    w.grad = None
    b.grad = None
    loss = ((w * x + b - y) ** 2).mean()
    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        updated_loss = ((w * x + b - y) ** 2).mean().item()

    # Check every step, even though we print only selected checkpoints.
    if updated_loss > previous_loss:
        increases.append(step)
    previous_loss = updated_loss

    if step in (10, 20, 50, 100):
        print(
            f"Step {step:3d} | weight={w.item():.4f} | "
            f"bias={b.item():.4f} | loss={updated_loss:.8f}"
        )

print("Steps where loss increased:", increases)


Step  10 | weight=3.2916 | bias=1.3505 | loss=0.15129718
Step  20 | weight=3.2232 | bias=1.5233 | loss=0.08241145
Step  50 | weight=3.0902 | bias=1.8073 | loss=0.01346400
Step 100 | weight=3.0199 | bias=1.9574 | loss=0.00065739
Steps where loss increased: []
